# Pipeline v10 visible — multimodelo y multisemilla

Esta notebook es **lineal, auditable y reproducible desde los datos crudos**. No ejecuta
otras notebooks ni lee predicciones de versiones anteriores.

Recorrido completo: base densa con ceros → features causales → clustering → LightGBM v3
con tres semillas → LightGBM v4 con tres semillas → Ridge → ensemble conservador → CSV.

Tiempo orientativo en la VM de 8 vCPU / 64 GB: **4 a 8 horas**.


## Mapa de modelos y combinación

| Rama | Unidad | Semillas | Papel en el resultado |
|---|---|---:|---:|
| v3 LightGBM | cliente-producto por 6 clusters | 109903, 102191, 314159 | 75% del campeón |
| v4 LightGBM | producto agregado | 109903, 102191, 314159 | 25% del campeón |
| Ridge | producto agregado | determinista | corrector limitado |

La corrección Ridge pesa 10%, pero su diferencia respecto del campeón se recorta a ±30%.
Por eso puede mover cada pronóstico, como máximo, aproximadamente ±3%.


In [1]:
# CELDA DE CONTROL: todo lo que se entrenará queda impreso antes de empezar
SEMILLAS_VISIBLES = [109903, 109927, 203999]
K_CLUSTERS_VISIBLE = 6
PESO_V3, PESO_V4 = 0.75, 0.25
PESO_RIDGE, TOPE_RIDGE = 0.10, 0.30
print("Semillas:", SEMILLAS_VISIBLES)
print("Clusters:", K_CLUSTERS_VISIBLE)
print("Ensemble campeón: v3=75%, v4=25%")
print("Corrector Ridge: peso=10%, diferencia recortada a ±30%")
print("IMPORTANTE: sólo se usarán archivos crudos del bucket; no submissions previas")


Semillas: [109903, 109927, 203999]
Clusters: 6
Ensemble campeón: v3=75%, v4=25%
Corrector Ridge: peso=10%, diferencia recortada a ±30%
IMPORTANTE: sólo se usarán archivos crudos del bucket; no submissions previas


# Parte A — LightGBM v3 cliente-producto

Esta parte conserva explícitamente todas las combinaciones cliente-producto-mes y completa
con `tn=0` las ventas ausentes, tal como exige la cátedra. Las features se desplazan para
que en cada fecha sólo utilicen información disponible hasta ese momento.


## 0) Setup en la nube (correr una vez)

Empezá con `modo_test=True` y `submit=False`. Cuando termine el smoke-test,
cambiá a `modo_test=False`. Esta versión usa una carpeta de salida nueva para
no mezclar checkpoints incompatibles con v2.


In [2]:
%pip install -q optuna dtaidistance lightgbm polars duckdb pandas numpy scipy scikit-learn joblib
# import os; os.environ['LABO3_BUCKET'] = '/ruta/al/bucket'

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _HAY_OPTUNA = True
except Exception:
    _HAY_OPTUNA = False

from dtaidistance import dtw
from joblib import Parallel, delayed
from sklearn.metrics import silhouette_score

N_CORES = os.cpu_count() or 4

# ¿está la lib C de dtaidistance? (en Linux/GCP viene en el wheel; acelera ~100x)
try:
    dtw.distance_fast(np.zeros(4), np.zeros(4), window=2)
    _C_OK = True
except Exception:
    _C_OK = False

try:
    dtw.warping_path(np.zeros(5), np.zeros(5), window=2)
    _WP_WINDOW = True
except TypeError:
    _WP_WINDOW = False


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 Path("/content/buckets/b1"), Path("/home/ds/buckets/b1")):
        if Path(cand).is_dir():
            return Path(cand)
    local = Path.home() / "Desktop" / "labo3-2026ba"
    if local.is_dir():
        return local
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


def escribir_atomico(fn, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    fn(tmp)
    tmp.replace(path)


def leer_json(path: Path, intentos=5, espera=1.0):
    for i in range(intentos):
        try:
            return json.loads(path.read_text())
        except (OSError, json.JSONDecodeError):
            if i == intentos - 1:
                raise
            time.sleep(espera)

## 1) Parámetros

In [4]:
PARAM = {
    "modo_test": False,              # LISTO PARA LA NUBE (run completo). Poner True = smoke test rapido en la VM.

    "horizonte": 2,
    "periodo_inferencia": 201912,    # parados aca -> predecir 202002
    "corte_train_final": 201910,     # ultimo periodo con clase (t+2 = 201912)
    "solo_productos_target": False,

    # ---- FE: 15 features (lista cerrada) ----
    "features": [
        "lag_1", "lag_2", "lag_3", "lag_6", "lag_10", "lag_12",
        "rmean_3", "rmean_6", "rmean_12", "rstd_3", "rstd_12",
        "nivel_relativo", "tendencia_3_12",
        "meses_consec_sin_compra", "ventas_ult12",
        "mes", "sin_mes", "cos_mes", "share_prod_en_cat3", "cat3", "brand",
    ],
    "cols_categoricas": ["cat3", "brand"],
    "eps": 1e-6,

    # ---- clustering k-means DTW + DBA ----
    "cl_escalado": "media",          # forma = tn/media (no tamaño)
    "cl_banda": 3,                   # Sakoe-Chiba (meses)
    "cl_corte": 201910,              # clustering causal: solo <= este periodo
    "cl_lista_k": [6],   # k candidatos (se elige el mejor)
    "cl_min_balance": 0.02,          # ningun cluster < 2% de las series
    "cl_muestra_k": 6000,            # series para ELEGIR k (silhouette)
    "cl_muestra_fit": 80000,         # series para AJUSTAR los centroides (None = todas)
    "cl_max_iter": 15, "cl_dba_iters": 2, "cl_tol": 0.01,

    # ---- LightGBM / Tweedie ----
    "max_bin": 1023,
    "num_threads": N_CORES,
    "objetivo_lgbm": "tweedie",
    "tweedie_vp_rango": (1.1, 1.6),

    # ---- Optuna (por cluster) ----
    "n_trials": 50,
    "semillas_ensemble": [109903, 109927, 203999],

    # ---- submit / magic multiplier ----
    "multiplicadores": [0.9, 1.0, 1.1],
    "clip_min": 0.0,
    "kaggle_competition": "labo-iii-2026-ba",
    "submit": True,   # sube las 3 automaticamente a Kaggle (requiere ~/.kaggle/kaggle.json)

    "semilla": 109903,
}

if PARAM["modo_test"]:
    PARAM.update({
        "n_productos_test": 15,
        "cl_lista_k": [2, 3],
        "cl_muestra_k": 250, "cl_muestra_fit": None,
        "n_trials": 4, "semillas_ensemble": [109903],
    })

BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / ("pipe_v10_balanceada_v3_test" if PARAM["modo_test"] else "pipe_v10_balanceada_v3")
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET={BUCKET}\nOUT={DIR_OUT}\ncores={N_CORES} optuna={_HAY_OPTUNA} "
      f"dtw_C={_C_OK} wp_window={_WP_WINDOW}")

BUCKET=/home/ds/buckets/b1
OUT=/home/ds/buckets/b1/pipe_v10_balanceada_v3
cores=8 optuna=True dtw_C=True wp_window=True


## 2) Datos crudos

In [5]:
def descargar(archivo):
    dst = DIR_RAW / archivo
    if dst.exists():
        return
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}"
    subprocess.run(["wget", url, "-O", str(dst)], check=True)


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    descargar(_a)

PROD_TARGET = set(
    pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")["product_id"].to_list()
)
print("productos a predecir:", len(PROD_TARGET))

productos a predecir: 780


## 3) Preprocesamiento — densa zero-fill cátedra (z601)

Grid ⟨cliente, producto, periodo⟩ con `tn=0` donde no hubo venta, dentro de la
vida del producto (min..max de venta, forzado a 201912 para los 780) **y** a
partir del primer periodo del cliente. Meses contiguos -> `shift(k)` = lag exacto.

In [6]:
PATH_PRE = DIR_OUT / "preprocesado.parquet"


def _construir_preprocesado():
    con = duckdb.connect(); con.execute("SET preserve_insertion_order=false")
    con.execute(f"""CREATE TABLE sellin AS
        SELECT CAST(customer_id AS INT) customer_id, CAST(product_id AS INT) product_id,
               CAST(periodo AS INT) periodo, CAST(tn AS DOUBLE) tn
        FROM read_csv_auto('{DIR_RAW/'sell-in.txt.gz'}')""")
    con.execute(f"""CREATE TABLE apredecir AS
        SELECT CAST(product_id AS INT) product_id FROM read_csv_auto('{DIR_RAW/'product_id_apredecir201912.txt'}')""")

    filtro = ""
    if PARAM["solo_productos_target"]:
        filtro = "WHERE product_id IN (SELECT product_id FROM apredecir)"
    elif PARAM["modo_test"]:
        filtro = (f"WHERE product_id IN (SELECT product_id FROM apredecir "
                  f"ORDER BY product_id LIMIT {PARAM['n_productos_test']})")

    con.execute(f"""CREATE TABLE base AS
        SELECT customer_id, product_id, periodo, SUM(tn) tn FROM sellin {filtro}
        GROUP BY customer_id, product_id, periodo""")
    con.execute("CREATE TABLE periodos AS SELECT DISTINCT periodo FROM base")
    con.execute("""CREATE TABLE vida_prod AS
        SELECT product_id, MIN(periodo) nace, MAX(periodo) muere FROM base GROUP BY product_id""")
    con.execute("""UPDATE vida_prod SET muere = 201912
        WHERE product_id IN (SELECT product_id FROM apredecir) AND muere < 201912""")
    con.execute("""CREATE TABLE primer_cli AS
        SELECT customer_id, MIN(periodo) nace_cli FROM base GROUP BY customer_id""")
    con.execute("""CREATE TABLE grid AS
        SELECT c.customer_id, v.product_id, p.periodo
        FROM vida_prod v JOIN periodos p ON p.periodo BETWEEN v.nace AND v.muere
        CROSS JOIN primer_cli c WHERE p.periodo >= c.nace_cli""")
    con.execute("""CREATE TABLE densa AS
        SELECT g.customer_id, g.product_id, g.periodo, COALESCE(b.tn, 0.0) tn
        FROM grid g LEFT JOIN base b USING (customer_id, product_id, periodo)""")

    prods = con.execute(f"""SELECT CAST(product_id AS INT) product_id, cat3, brand
        FROM read_csv_auto('{DIR_RAW/'tb_productos.txt'}')""").df()
    df = con.execute("SELECT * FROM densa ORDER BY customer_id, product_id, periodo").pl()
    con.close()

    df = df.join(pl.from_pandas(prods), on="product_id", how="left")
    MULT = 100_000
    return df.with_columns(
        (pl.col("customer_id").cast(pl.Int64) * MULT + pl.col("product_id").cast(pl.Int64)).alias("agrupa_id")
    )


if PATH_PRE.exists():
    print("[preprocesado] RESUME"); df_pre = pl.read_parquet(PATH_PRE)
else:
    df_pre = _construir_preprocesado()
    escribir_atomico(lambda t: df_pre.write_parquet(t), PATH_PRE)
print("preprocesado:", df_pre.shape, "| series:", df_pre["agrupa_id"].n_unique())

[preprocesado] RESUME
preprocesado: (17173448, 7) | series: 722457


## 4) Escalado + Feature Engineering causal

Todas las variables usan información disponible hasta t. `lag_10` observado en
diciembre corresponde a febrero del mismo año: es el comparable estacional directo
para el objetivo febrero de 2020.


In [7]:
PATH_FE = DIR_OUT / "features.parquet"


def _fe(df: pl.DataFrame) -> pl.DataFrame:
    g = "agrupa_id"; eps = PARAM["eps"]
    df = df.sort([g, "periodo"])
    df = df.with_columns(
        pl.col("tn").cum_sum().over(g).alias("_cs"),
        (pl.int_range(pl.len()).over(g) + 1).alias("_n"),
        pl.int_range(pl.len()).over(g).alias("_rn"),
    ).with_columns((pl.col("_cs") / pl.col("_n")).alias("s"))

    tn1 = pl.col("tn").shift(1).over(g)
    df = df.with_columns([
        pl.col("tn").shift(1).over(g).alias("lag_1"),
        pl.col("tn").shift(2).over(g).alias("lag_2"),
        pl.col("tn").shift(3).over(g).alias("lag_3"),
        pl.col("tn").shift(6).over(g).alias("lag_6"),
        pl.col("tn").shift(10).over(g).alias("lag_10"),
        pl.col("tn").shift(12).over(g).alias("lag_12"),
        tn1.rolling_mean(3, min_samples=1).alias("rmean_3"),
        tn1.rolling_mean(6, min_samples=1).alias("rmean_6"),
        tn1.rolling_mean(12, min_samples=1).alias("rmean_12"),
        tn1.rolling_std(3, min_samples=2).alias("rstd_3"),
        tn1.rolling_std(12, min_samples=2).alias("rstd_12"),
        (pl.col("tn") > 0).cast(pl.Int32).rolling_sum(12, min_samples=1).over(g).shift(1).over(g).alias("ventas_ult12"),
        (pl.col("periodo") % 100).alias("mes"),
        ((2 * np.pi * (pl.col("periodo") % 100) / 12).sin()).alias("sin_mes"),
        ((2 * np.pi * (pl.col("periodo") % 100) / 12).cos()).alias("cos_mes"),
    ])
    df = df.with_columns([
        (pl.col("lag_1") / (pl.col("s") + eps)).alias("nivel_relativo"),
        (pl.col("rmean_3") / (pl.col("rmean_12") + eps)).alias("tendencia_3_12"),
        pl.when(pl.col("tn") > 0).then(pl.col("_rn")).otherwise(None).forward_fill().over(g).alias("_uv"),
    ])
    df = df.with_columns(
        (pl.col("_rn") - pl.col("_uv").fill_null(-1)).alias("meses_consec_sin_compra")
    )
    df = df.with_columns([
        pl.col("tn").sum().over(["product_id", "periodo"]).alias("_tn_prod"),
        pl.col("tn").sum().over(["cat3", "periodo"]).alias("_tn_cat3"),
    ]).with_columns(
        (pl.col("_tn_prod") / (pl.col("_tn_cat3") + eps)).alias("share_prod_en_cat3")
    )
    # clase (unico futuro) y su escalado
    df = df.with_columns(pl.col("tn").shift(-PARAM["horizonte"]).over(g).alias("clase_raw"))
    df = df.with_columns(
        pl.when(pl.col("s") > eps).then(pl.col("clase_raw") / pl.col("s")).otherwise(0.0).alias("clase_scaled")
    )
    keep = (["agrupa_id", "customer_id", "product_id", "periodo", "tn", "s",
             "clase_raw", "clase_scaled"] + PARAM["features"])
    return df.select([c for c in keep if c in df.columns])


if PATH_FE.exists():
    print("[features] RESUME"); df_fe = pl.read_parquet(PATH_FE)
else:
    df_fe = _fe(df_pre)
    escribir_atomico(lambda t: df_fe.write_parquet(t), PATH_FE)
print("features:", df_fe.shape, "|", len(PARAM["features"]), "features")

[features] RESUME
features: (17173448, 29) | 21 features


## 5) Clustering k-means DTW + DBA  (motor de Rosario, mejorado)

Serie por par = `tn/media` desde su primera venta, sólo `<= cl_corte` (causal).
Mejoras: `use_pruning`, asignación paralela a todos los núcleos, elección de k
por silhouette+balance sobre una muestra, y **todas** las series reciben cluster
(los centroides finales se ajustan sobre `cl_muestra_fit` y se asigna el resto).

In [8]:
PATH_CL = DIR_OUT / "clusters.parquet"
BANDA = PARAM["cl_banda"]


def _banda(a, b):
    return max(int(BANDA), abs(len(a) - len(b)))


def _d(a, b):
    if _C_OK:
        d = dtw.distance_fast(a, b, window=_banda(a, b))
    else:
        d = dtw.distance(a, b, window=_banda(a, b), use_c=False)
    return d if np.isfinite(d) else np.inf


def _sanear(D):
    """reemplaza distancias no finitas (banda infactible) por un valor grande finito."""
    mal = ~np.isfinite(D)
    if mal.any():
        fin = D[~mal]
        D[mal] = (fin.max() * 10.0) if fin.size else 1.0
    return D


def _dvec(s, centros):
    """distancias de s a cada centro, saneadas (sin inf/nan)."""
    return _sanear(np.array([_d(s, c) for c in centros], dtype=np.float64))


def _serie(x: np.ndarray):
    """tn/media desde la primera venta; None si nunca vendió en <= corte."""
    nz = np.nonzero(x > 0)[0]
    if len(nz) == 0:
        return None
    x = np.ascontiguousarray(x[nz[0]:], dtype=np.float64)
    m = x.mean()
    return x / m if m > PARAM["eps"] else x


def _asignar(series, centros):
    def uno(s):
        return int(np.argmin(_dvec(s, centros)))
    return np.array(Parallel(n_jobs=N_CORES, prefer="threads", batch_size=256)(
        delayed(uno)(s) for s in series))


def _dba(miembros, centro, iters):
    centro = np.ascontiguousarray(centro, dtype=np.float64)
    if not miembros:
        return centro
    T = len(centro)
    for _ in range(iters):
        acum = np.zeros(T); cnt = np.zeros(T)
        for s in miembros:
            w = _banda(centro, s)
            path = dtw.warping_path(centro, s, window=w) if _WP_WINDOW else dtw.warping_path(centro, s)
            for i, j in path:
                acum[i] += s[j]; cnt[i] += 1.0
        centro = np.where(cnt > 0, acum / np.maximum(cnt, 1.0), centro)
        centro = np.ascontiguousarray(centro, dtype=np.float64)
    return centro


def _kmeanspp(series, k, rng):
    centros = [series[int(rng.integers(len(series)))].copy()]
    dmin = _sanear(np.array([_d(s, centros[0]) for s in series], dtype=np.float64))
    for _ in range(1, k):
        p = dmin ** 2; tot = p.sum()
        idx = int(rng.integers(len(series))) if not np.isfinite(tot) or tot <= 0 \
            else int(rng.choice(len(series), p=p / tot))
        centros.append(series[idx].copy())
        dmin = np.minimum(dmin, _sanear(np.array([_d(s, centros[-1]) for s in series], dtype=np.float64)))
    return centros


def _kmeans_dtw(series, k, semilla):
    rng = np.random.default_rng(semilla)
    centros = _kmeanspp(series, k, rng)
    lab = np.full(len(series), -1)
    for it in range(PARAM["cl_max_iter"]):
        lab_new = _asignar(series, centros)
        cambios = int((lab_new != lab).sum()); lab = lab_new
        for j in range(k):  # cluster vacio -> re-seed con la serie mas lejana
            if not np.any(lab == j):
                lab[int(rng.integers(len(series)))] = j
        centros = [_dba([series[i] for i in np.flatnonzero(lab == j)], centros[j], PARAM["cl_dba_iters"])
                   for j in range(k)]
        if cambios / len(series) < PARAM["cl_tol"]:
            break
    return lab, centros


def _matriz(series):
    n = len(series); D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = _d(series[i], series[j])
    D = _sanear(D)
    np.fill_diagonal(D, 0.0)
    return D


def _construir_clusters(df: pl.DataFrame) -> pl.DataFrame:
    hist = df.filter(pl.col("periodo") <= PARAM["cl_corte"])
    piv = hist.group_by("agrupa_id").agg(
        pl.col("tn").sort_by("periodo").alias("serie"),
        pl.col("tn").sum().alias("tn_total"))
    ids = piv["agrupa_id"].to_list()
    series_raw = [np.asarray(s, dtype=np.float64) for s in piv["serie"].to_list()]
    tn_tot = np.asarray(piv["tn_total"].to_list())
    series = [_serie(x) for x in series_raw]
    validos = [i for i, s in enumerate(series) if s is not None]
    rng = np.random.default_rng(PARAM["semilla"])

    K = 6
    print("  -> k fijo = 6 (seleccionado experimentalmente)")

    # (b) ajustar centroides finales sobre cl_muestra_fit (o todas) y asignar TODAS
    pool = validos
    if PARAM["cl_muestra_fit"] and len(validos) > PARAM["cl_muestra_fit"]:
        orden = np.argsort(-tn_tot[validos])[:PARAM["cl_muestra_fit"]]
        pool = [validos[i] for i in orden]
    _, centros = _kmeans_dtw([series[i] for i in pool], K, PARAM["semilla"])

    lab_todos = np.zeros(len(ids), dtype=int)
    lab_todos[validos] = _asignar([series[i] for i in validos], centros)
    # series sin venta en <=corte -> cluster mayoritario
    if len(validos) < len(ids):
        mayor = int(np.bincount(lab_todos[validos]).argmax())
        for i in range(len(ids)):
            if series[i] is None:
                lab_todos[i] = mayor

    return pl.DataFrame({"agrupa_id": ids, "cluster_id": lab_todos.astype(np.int32)})


if PATH_CL.exists():
    print("[clusters] RESUME"); df_cl = pl.read_parquet(PATH_CL)
else:
    df_cl = _construir_clusters(df_fe)
    escribir_atomico(lambda t: df_cl.write_parquet(t), PATH_CL)
print("clusters:", df_cl.group_by("cluster_id").len().sort("cluster_id").to_dicts())

df_fe = df_fe.join(df_cl, on="agrupa_id", how="left").with_columns(
    pl.col("cluster_id").fill_null(df_cl["cluster_id"].min()).cast(pl.Int32))

[clusters] RESUME
clusters: [{'cluster_id': 0, 'len': 94338}, {'cluster_id': 1, 'len': 52790}, {'cluster_id': 2, 'len': 18892}, {'cluster_id': 3, 'len': 19082}, {'cluster_id': 4, 'len': 75707}, {'cluster_id': 5, 'len': 452704}]


## 6) WAPE a nivel producto + validación diciembre → febrero

Los folds usan como origen un diciembre cuyo target t+2 (febrero) ya es observable.
Diciembre de 2019 queda reservado para la inferencia de febrero de 2020.


In [9]:
def periodo_mas_h(p, h):
    y, m = divmod(int(p), 100)
    tot = y * 12 + (m - 1) + h
    return (tot // 12) * 100 + (tot % 12) + 1


def splits_febreros(periodos, h):
    pset = set(map(int, periodos))
    return [p for p in sorted(pset)
            if periodo_mas_h(p, h) in pset and periodo_mas_h(p, h) % 100 == 2]


assert periodo_mas_h(PARAM["periodo_inferencia"], PARAM["horizonte"]) == 202002


def pesos_recencia(periodos, half_life):
    p = np.asarray(periodos, dtype=int)
    idx = (p // 100) * 12 + (p % 100) - 1
    edad = idx.max() - idx
    return np.power(0.5, edad / max(float(half_life), 1.0))


def wape_producto(df_val: pd.DataFrame) -> float:
    d = df_val[df_val["product_id"].isin(PROD_TARGET)]
    if len(d) == 0:
        return float("nan")
    gg = d.groupby("product_id").agg(actual=("clase_raw", "sum"), fcst=("pred_tn", "sum"))
    den = gg["actual"].sum()
    return float("nan") if den <= 0 else float(np.abs(gg["actual"] - gg["fcst"]).sum() / den)


## 7) Optuna + entrenamiento final por cluster

Optuna ajusta LightGBM, la intensidad de recencia y el peso entre el modelo y el
baseline estacional. La métrica continúa siendo WAPE agregado por producto.


In [10]:
FEATS = PARAM["features"]
CATS = [c for c in PARAM["cols_categoricas"] if c in FEATS]


def _pd_cluster(k):
    d = df_fe.filter(pl.col("cluster_id") == k).to_pandas()
    for c in CATS:
        d[c] = d[c].astype("category")
    return d


def _params(trial, seed):
    p = {"objective": PARAM["objetivo_lgbm"], "metric": "tweedie", "max_bin": PARAM["max_bin"],
         "num_threads": PARAM["num_threads"], "verbosity": -1, "boosting_type": "gbdt", "seed": seed}
    if PARAM["objetivo_lgbm"] == "tweedie":
        lo, hi = PARAM["tweedie_vp_rango"]
        p["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", lo, hi)
    p.update({
        "num_leaves": trial.suggest_int("num_leaves", 16, 192),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": 1,
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
    })
    return p


PARAMS_FIJOS_V3 = {0: {'cluster': 0, 'wape_val': 0.19176948222935203, 'params': {'tweedie_variance_power': 1.2156846132786492, 'num_leaves': 159, 'max_depth': 10, 'learning_rate': 0.05884069479188206, 'n_estimators': 1226, 'min_child_samples': 109, 'feature_fraction': 0.6107664020543767, 'bagging_fraction': 0.674796076489137, 'reg_alpha': 0.003028452307824398, 'reg_lambda': 1.2994533978532667e-08, 'blend_lgb': 0.9875103849023539, 'half_life': 24.681169042960395}, 'n_filas': 10767416}, 1: {'cluster': 1, 'wape_val': 0.5701435830056607, 'params': {'tweedie_variance_power': 1.3602162904011736, 'num_leaves': 138, 'max_depth': 8, 'learning_rate': 0.019268989493488528, 'n_estimators': 1185, 'min_child_samples': 25, 'feature_fraction': 0.7490629567355026, 'bagging_fraction': 0.8672944027738667, 'reg_alpha': 0.00016311198167623975, 'reg_lambda': 0.051959141841561106, 'blend_lgb': 0.9109615087553508, 'half_life': 17.455493299573924}, 'n_filas': 918490}, 2: {'cluster': 2, 'wape_val': 0.8134278024237791, 'params': {'tweedie_variance_power': 1.420603156379721, 'num_leaves': 36, 'max_depth': 11, 'learning_rate': 0.008429466093860728, 'n_estimators': 417, 'min_child_samples': 212, 'feature_fraction': 0.9445280430328726, 'bagging_fraction': 0.9433387617648121, 'reg_alpha': 0.013196257962474103, 'reg_lambda': 3.0949044126812408e-06, 'blend_lgb': 0.942805420826836, 'half_life': 14.29910801653533}, 'n_filas': 1740409}, 3: {'cluster': 3, 'wape_val': 0.47125710440790836, 'params': {'tweedie_variance_power': 1.3589710942692232, 'num_leaves': 178, 'max_depth': 11, 'learning_rate': 0.008303436308796474, 'n_estimators': 111, 'min_child_samples': 194, 'feature_fraction': 0.7219153050762825, 'bagging_fraction': 0.6190814608007674, 'reg_alpha': 1.0262376574869856e-07, 'reg_lambda': 0.005606981438758368, 'blend_lgb': 0.8667872409242223, 'half_life': 18.62265896445487}, 'n_filas': 1923234}, 4: {'cluster': 4, 'wape_val': 1.1210292611148993, 'params': {'tweedie_variance_power': 1.5987440980448056, 'num_leaves': 78, 'max_depth': 11, 'learning_rate': 0.1881826879205455, 'n_estimators': 1357, 'min_child_samples': 181, 'feature_fraction': 0.6884370233526224, 'bagging_fraction': 0.793637876061271, 'reg_alpha': 0.0003578573563730557, 'reg_lambda': 0.01707081954956836, 'blend_lgb': 0.8652632210514345, 'half_life': 13.924921400541052}, 'n_filas': 922254}, 5: {'cluster': 5, 'wape_val': 0.6924784068976495, 'params': {'tweedie_variance_power': 1.3074670249929616, 'num_leaves': 161, 'max_depth': 9, 'learning_rate': 0.14159663060805436, 'n_estimators': 489, 'min_child_samples': 168, 'feature_fraction': 0.7599209517903601, 'bagging_fraction': 0.6512985038122863, 'reg_alpha': 2.4277013069343715e-05, 'reg_lambda': 4.878729222093479e-08, 'blend_lgb': 0.9456394288010939, 'half_life': 11.655957900272366}, 'n_filas': 901645}}

def _optuna_cluster(d, k):
    return PARAMS_FIJOS_V3[int(k)]

def _entrenar_predecir(d, hiper):
    base = {"objective": PARAM["objetivo_lgbm"], "metric": "tweedie", "max_bin": PARAM["max_bin"],
            "num_threads": PARAM["num_threads"], "verbosity": -1, "boosting_type": "gbdt"}
    tuned = dict(hiper["params"])
    blend_lgb = float(tuned.pop("blend_lgb", 1.0))
    half_life = float(tuned.pop("half_life", 18.0))
    base.update(tuned)
    tr = d[d["clase_raw"].notna()]
    inf = d[d["periodo"] == PARAM["periodo_inferencia"]].copy()
    if len(inf) == 0:
        return pd.DataFrame(columns=["product_id", "pred_tn"])
    preds = []
    for s in PARAM["semillas_ensemble"]:
        m = lgb.LGBMRegressor(**{**base, "seed": s})
        m.fit(tr[FEATS], tr["clase_scaled"], categorical_feature=CATS,
              sample_weight=pesos_recencia(tr["periodo"].to_numpy(), half_life))
        preds.append(np.maximum(m.predict(inf[FEATS]), 0.0))
    pred_lgb = np.mean(preds, axis=0) * inf["s"].to_numpy()
    pred_yoy = np.maximum(inf["lag_10"].fillna(0.0).to_numpy(), 0.0)
    inf["pred_tn"] = np.maximum(blend_lgb * pred_lgb + (1.0 - blend_lgb) * pred_yoy,
                                PARAM["clip_min"])
    return inf[["product_id", "pred_tn"]]


PATH_PRED = DIR_OUT / "pred_base.parquet"
DIR_PREDK = DIR_OUT / "pred"

if PATH_PRED.exists():
    print("[pred_base] RESUME"); pred_base = pl.read_parquet(PATH_PRED).to_pandas()
else:
    # bucle por cluster con checkpoint POR cluster (spot-friendly): cada cluster deja
    # su optuna (optuna/cluster_k.json) y su prediccion (pred/cluster_k.parquet); una
    # preemption solo pierde el cluster en curso, no los ya terminados.
    for k in sorted(df_fe["cluster_id"].unique().to_list()):
        pth_k = DIR_PREDK / f"cluster_{k}.parquet"
        if pth_k.exists():
            print(f"  [cluster {k}] RESUME (pred ya existe)")
            continue
        d = _pd_cluster(k)
        if d["clase_raw"].notna().sum() == 0:
            escribir_atomico(lambda t: pl.DataFrame(
                {"product_id": pl.Series([], dtype=pl.Int64),
                 "pred_tn": pl.Series([], dtype=pl.Float64)}).write_parquet(t), pth_k)
            continue
        hip = _optuna_cluster(d, k)
        pr = _entrenar_predecir(d, hip)
        escribir_atomico(lambda t, pr=pr: pl.from_pandas(pr).write_parquet(t), pth_k)
        print(f"  cluster {k}: filas={hip['n_filas']:>8}  wape_val={hip['wape_val']:.4f}")

    partes = [pl.read_parquet(f).to_pandas() for f in sorted(DIR_PREDK.glob("cluster_*.parquet"))]
    partes = [p for p in partes if len(p)]
    pred_base = pd.concat(partes, ignore_index=True).groupby("product_id", as_index=False)["pred_tn"].sum()
    escribir_atomico(lambda t: pl.from_pandas(pred_base).write_parquet(t), PATH_PRED)

# resumen desde los checkpoints de optuna (robusto a resume)
resumen = [(h["cluster"], h["n_filas"], round(h["wape_val"], 4))
           for h in (leer_json(j) for j in sorted((DIR_OUT / "optuna").glob("cluster_*.json")))]
if resumen:
    print("resumen (cluster, filas, wape_val):", resumen)
    print("wape_val ponderado:", round(np.average([r[2] for r in resumen],
                                                  weights=[r[1] for r in resumen]), 4))

[pred_base] RESUME


## 8) Submissions calibradas

Primero probá `submission_x1.0.csv`. Los escenarios 0.9 y 1.1 permiten medir
si el leaderboard favorece una corrección global.


In [11]:
apr = pd.read_csv(DIR_RAW / "product_id_apredecir201912.txt", sep="\t")[["product_id"]]
base_sub = apr.merge(pred_base.rename(columns={"pred_tn": "tn"}), on="product_id", how="left")
base_sub["tn"] = base_sub["tn"].fillna(PARAM["clip_min"])

for mult in [1.0]:
    sub = base_sub.copy()
    sub["tn"] = np.maximum(sub["tn"] * mult, PARAM["clip_min"])
    ruta = DIR_OUT / f"submission_x{mult}.csv"
    escribir_atomico(lambda t, s=sub: s.to_csv(t, index=False), ruta)
    print(f"  submission ×{mult}: filas={len(sub)} tn_total={sub['tn'].sum():.1f} -> {ruta.name}")

if PARAM["submit"]:
    for mult in [1.0]:
        flag = DIR_OUT / "submits" / f"x{mult}.done"
        if flag.exists():
            print(f"  submit ×{mult} ya hecho -> se saltea"); continue
        ruta = DIR_OUT / f"submission_x{mult}.csv"
        try:
            r = subprocess.run(
                ["kaggle", "competitions", "submit", "-c", PARAM["kaggle_competition"],
                 "-f", str(ruta), "-m", f"pipe_v2 x{mult}"],
                capture_output=True, text=True)
            print(f"  submit ×{mult}: rc={r.returncode} {(r.stdout or r.stderr).strip()[:200]}")
            if r.returncode == 0:
                escribir_atomico(lambda t: Path(t).write_text(time.strftime("%Y-%m-%d %H:%M:%S")), flag)
        except Exception as e:
            print(f"  submit ×{mult} fallo (subir a mano el csv): {e}")

print("\nOK. Submissions en", DIR_OUT, "(submission_x*.csv)")

  submission ×1.0: filas=780 tn_total=29022.1 -> submission_x1.0.csv
  submit ×1.0 ya hecho -> se saltea

OK. Submissions en /home/ds/buckets/b1/pipe_v10_balanceada_v3 (submission_x*.csv)


In [12]:
# CHECKPOINT EXPLÍCITO DE LA RAMA v3 (creado en esta misma ejecución)
PATH_V3_FINAL = DIR_OUT / "submission_x1.0.csv"
if not PATH_V3_FINAL.exists():
    raise FileNotFoundError(f"La rama v3 no generó {PATH_V3_FINAL}")
pred_v3_visible = pd.read_csv(PATH_V3_FINAL).sort_values("product_id").reset_index(drop=True)
print("v3 lista:", PATH_V3_FINAL)
print("filas=", len(pred_v3_visible), "tn total=", pred_v3_visible["tn"].sum())


v3 lista: /home/ds/buckets/b1/pipe_v10_balanceada_v3/submission_x1.0.csv
filas= 780 tn total= 29022.091442454624


# Parte B — LightGBM v4 agregado

Se vuelve a partir de los datos crudos. La configuración ganadora queda escrita en las
celdas siguientes: baseline `ma_pond`, esquema `B_lgbm_nivel`, tres semillas y parámetros
LightGBM congelados. No se ejecuta Optuna durante la entrega.


In [13]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.linear_model import Ridge

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_EXP = BUCKET / "exp_residuo"
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET: {BUCKET}")
print(f"salida: {RUTA_EXP}")

BUCKET: /home/ds/buckets/b1
salida: /home/ds/buckets/b1/exp_residuo


## 1 — Palancas

In [14]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ── Datos ────────────────────────────────────────────────────────────
    'solo_productos_target': True,   # los 780 que se entregan
    'muestra_productos': None,       # None = todos. Un numero para probar el pipe rapido.
    'horizonte': 2,
    'max_lags': 12,

    # ── Particion (misma que el pipe, para poder comparar) ───────────────
    'meses_train': rango_meses(201701, 201710), # targets observables antes de dic-2017
    'meses_val':   [201712],               # dic-2017 -> feb-2018
    'meses_test':  [201812],               # dic-2018 -> feb-2019
    'reentrenar_con_val_para_test': True,

    # ── EL BASELINE ──────────────────────────────────────────────────────
    # Sobre que se calcula el residuo. 'auto' prueba todos y elige por VALIDACION.
    #   'tn0'      -> repetir el ultimo mes (el naive)
    #   'ma3/6/12' -> promedio movil de N meses
    #   'ma_pond'  -> 0.5*tn0 + 0.3*tn1 + 0.2*tn2, pesos fijos
    #   'lineal'   -> Ridge sobre los lags: el promedio movil con pesos APRENDIDOS.
    #                 Es la version formal de la regresion stepwise que dio 0,23.
    'baseline': 'ma_pond',

    # ── EL ESQUEMA ───────────────────────────────────────────────────────
    # 'auto' compara los seis y sigue con el mejor en validacion.
    # Forzarlo sirve para aislar un esquema y compararlo en el leaderboard.
    #   'A_baseline' | 'B_lgbm_nivel' | 'C_lineal_nivel'
    #   'D_lgbm_residuo' | 'E_lineal_mas_lgbm' | 'F_lgbm_hojas_lineales'
    'esquema': 'B_lgbm_nivel',

    # ── Optuna sobre el esquema ganador ──────────────────────────────────
    'n_trials': 50,
    'techo_arboles': 800,

    # ── Ridge ────────────────────────────────────────────────────────────
    # Regularizacion de la parte lineal. Con lags muy correlacionados entre si
    # (que es el caso) sin regularizar los coeficientes se vuelven inestables.
    'ridge_alpha': 1.0,

    # ── Entrega ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semillas_ensemble': [109903, 109927, 203999],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-ba',
    'submit': True,
    'submit_ensembles': True,  # activar sólo después de mirar el score v4 puro
    'mensaje_submit': None,

    'semilla': 109903,
    'sufijo': 'v10_balanceada_v4',
}

H = PARAM['horizonte']
L = PARAM['max_lags']

EXPERIMENTO = (f"residuo_p_{L}lags_base-{PARAM['baseline']}_esq-{PARAM['esquema']}"
               f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
               f"_test{PARAM['meses_test'][0]}"
               + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"EXPERIMENTO: {EXPERIMENTO}")
print(f"carpeta    : {DIR_OUT.relative_to(BUCKET)}")

EXPERIMENTO: residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v10_balanceada_v4
carpeta    : exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v10_balanceada_v4


## 2 — Preprocesamiento y features

Panel producto-mes densificado dentro de la vida de cada producto, con lags, promedios
móviles, shares en los tres niveles de categoría, deltas de share e índices. Todo
causal: cada feature de la fila `t` usa sólo información hasta `t`.

In [15]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
target_ids = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")["product_id"].to_list()

if PARAM['solo_productos_target']:
    sell = sell.filter(pl.col("product_id").is_in(target_ids))
if PARAM['muestra_productos']:
    _top = (sell.group_by("product_id").agg(pl.col("tn").sum().alias("t"))
                .sort("t", descending=True).head(PARAM['muestra_productos'])["product_id"])
    sell = sell.filter(pl.col("product_id").is_in(_top.to_list()))

print(f"sell-in: {sell.height:,} filas · {sell['product_id'].n_unique()} productos "
      f"· {sell['customer_id'].n_unique()} clientes")

# ── Panel producto-mes: se colapsa la dimension cliente, que es la misma
#    agregacion que hace la metrica de la competencia antes de medir.
panel = (sell.group_by(["product_id", "periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_tn").sum().alias("req_tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("customer_id").n_unique().alias("n_clientes"),
                  pl.col("plan_precios_cuidados").max().alias("precios_cuidados"))
             .with_columns((((pl.col("periodo") // 100) * 12)
                            + (pl.col("periodo") % 100)).alias("m")))

vida = panel.group_by("product_id").agg(
    pl.col("m").min().alias("m_nace"),
    pl.col("m").max().alias("m_muere")
)
# Los productos de entrega deben tener fila en 201912 aunque hayan dejado de vender.
M_DIC2019 = (2019 * 12) + 12
vida = vida.with_columns(
    pl.when(pl.col("product_id").is_in(target_ids))
      .then(pl.max_horizontal(pl.col("m_muere"), pl.lit(M_DIC2019)))
      .otherwise(pl.col("m_muere")).alias("m_muere")
)
grilla = (vida.with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").select("product_id", "m"))

panel = (grilla.join(panel.drop("periodo"), on=["product_id", "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0), pl.col("req_tn").fill_null(0.0),
                             pl.col("req_qty").fill_null(0), pl.col("n_clientes").fill_null(0),
                             pl.col("precios_cuidados").fill_null(0))
               .join(vida, on="product_id", how="left")
               .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                     on="product_id", how="left")
               .with_columns(
                   ((((pl.col("m") - 1) // 12) * 100) + ((pl.col("m") - 1) % 12) + 1)
                     .alias("periodo"),
                   pl.when(pl.col("m") >= pl.col("m_nace"))
                     .then(pl.col("m") - pl.col("m_nace")).otherwise(-1).alias("edad"))
               .sort(["product_id", "m"]))

print(f"panel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")

# ── Totales para los shares (todos del mes t: contexto, no futuro) ──────
for niv in ("cat1", "cat2", "cat3"):
    t = panel.group_by([niv, "m"]).agg(pl.col("tn").sum().alias(f"tn_{niv}"))
    panel = panel.join(t, on=[niv, "m"], how="left")
mercado = panel.group_by("m").agg(pl.col("tn").sum().alias("tn_mercado"))
panel = panel.join(mercado, on="m", how="left")


def div_segura(num, den, nombre):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then(pl.col(num) / pl.col(den)).otherwise(0.0).alias(nombre))


SHARES = [f"sh_{n}" for n in ("cat1", "cat2", "cat3", "mercado")]
panel = panel.with_columns([div_segura("tn", f"tn_{n}", f"sh_{n}")
                            for n in ("cat1", "cat2", "cat3", "mercado")])

# ── Lags, promedios moviles, deltas e indices ───────────────────────────
df = panel.sort(["product_id", "m"]).with_columns(
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over("product_id").alias(f"tn_ma{w}") for w in (3, 6, 12)],
    *[pl.col(s).shift(k).over("product_id").alias(f"{s}_lag{k}")
      for s in SHARES for k in (1, 2, 3)],
    *[pl.col(s).rolling_mean(3).over("product_id").alias(f"{s}_ma3") for s in SHARES],
    pl.col("n_clientes").shift(1).over("product_id").alias("n_clientes_lag1"),
    pl.col("n_clientes").rolling_mean(3).over("product_id").alias("n_clientes_ma3"),
    pl.col("req_qty").shift(1).over("product_id").alias("qty_lag1"),
    pl.col("tn").cum_max().over("product_id").alias("tn_pico_hasta_aca"),
    (pl.col("tn") > 0).cast(pl.Int8).alias("vendio"),
)

df = df.with_columns(
    *[(pl.col(s) - pl.col(f"{s}_lag1")).alias(f"{s}_d1") for s in SHARES],
    *[(pl.col(s) - pl.col(f"{s}_ma3")).alias(f"{s}_dma3") for s in SHARES],
    (pl.col("tn") - pl.col("tn_ma3")).alias("tn_dma3"),
    pl.col("vendio").rolling_mean(6).over("product_id").alias("frac_venta_6"),
    (pl.col("periodo") % 100).alias("mes_del_anio"),
    (pl.col("edad").is_between(0, 6)).cast(pl.Int8).alias("es_nuevo"),
)


def indice(num, den, nombre, techo=10.0):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then((pl.col(num) / pl.col(den)).clip(0.0, techo))
              .otherwise(pl.lit(None, dtype=pl.Float64)).alias(nombre))


df = df.with_columns(
    indice("tn", "tn_lag1", "idx_tn_mom"),
    indice("tn", "tn_ma3", "idx_tn_vs_ma3"),
    indice("tn", "tn_pico_hasta_aca", "idx_vs_pico"),
    indice("n_clientes", "n_clientes_lag1", "idx_clientes_mom"),
    indice("req_qty", "qty_lag1", "idx_qty_mom"),
)

# ── El target ───────────────────────────────────────────────────────────
df = df.sort(["product_id", "m"]).with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("clase_tn"),
    ((((pl.col("m") + H - 1) // 12) * 100) + ((pl.col("m") + H - 1) % 12) + 1)
      .alias("periodo_objetivo"),
)

CATS = ["cat1", "cat2", "cat3", "brand"]
df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                      for c in CATS])

NO_FEAT = {"product_id", "periodo", "m", "m_nace", "m_muere", "clase_tn",
           "periodo_objetivo"}
FEATURES = [c for c in df.columns if c not in NO_FEAT]

print(f"features: {len(FEATURES)}   filas: {df.height:,}   [{time.time()-t0:.0f}s]")
print(f"con target: {int(df['clase_tn'].is_not_null().sum()):,}")

sell-in: 2,293,481 filas · 780 productos · 597 clientes
panel: 22,375 filas · 780 productos
features: 72   filas: 22,375   [1s]
con target: 20,815


/tmp/ipykernel_3755/1965495718.py:41: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("m").select("product_id", "m"))


## 3 — Partición y control de leakage

In [16]:
sup = df.filter(pl.col("clase_tn").is_not_null())
periodos_sup = sorted(sup["periodo"].unique().to_list())
MESES_TRAIN = [m for m in PARAM['meses_train'] if m in periodos_sup]
MESES_VAL   = [m for m in PARAM['meses_val'] if m in periodos_sup]
MESES_TEST  = [m for m in PARAM['meses_test'] if m in periodos_sup]
MESES_INFER = sorted(df.filter(pl.col("clase_tn").is_null())["periodo"].unique().to_list())[-H:]
infer = df.filter(pl.col("periodo").is_in(MESES_INFER))

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


def a_m(p):
    return (p // 100) * 12 + (p % 100)


print("CONTROL DE LEAKAGE")
print("=" * 74)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                     (MESES_VAL, MESES_TEST, "val", "test")):
    g = a_m(min(b)) - a_m(max(a))
    chk(g >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {g} >= horizonte {H}")
if PARAM['reentrenar_con_val_para_test']:
    g = a_m(min(MESES_TEST)) - a_m(max(MESES_TRAIN + MESES_VAL))
    chk(g >= H, f"gap (train+val) -> test = {g} >= {H}")
chk(max(MESES_TRAIN) < min(MESES_VAL) and max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
chk("m_muere" not in FEATURES, "m_muere (dato del futuro) no es feature")
chk(not (set(FEATURES) & {"clase_tn", "periodo_objetivo"}), "el target no es feature")

# el shift del target, verificado fila por fila en la serie mas larga
_u = sup.group_by("product_id").agg(pl.len().alias("n")).sort("n", descending=True).head(1)
_s = df.filter(pl.col("product_id") == _u["product_id"][0]).sort("m")
_tn, _cl = _s["tn"].to_list(), _s["clase_tn"].to_list()
_mal = [i for i in range(len(_tn) - H)
        if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
chk(not _mal, f"clase_tn[i] == tn[i+{H}] en el producto {_u['product_id'][0]} "
              f"({len(_tn)} meses, {len(_mal)} discrepancias)")

print("=" * 74)
if errores:
    raise RuntimeError(f"Leakage: {errores}")
print(f"TRAIN {len(MESES_TRAIN)} meses ({sup.filter(pl.col('periodo').is_in(MESES_TRAIN)).height:,} filas)"
      f" · VAL {MESES_VAL} · TEST {MESES_TEST} · INFER {MESES_INFER}")

CONTROL DE LEAKAGE
  [ok   ] gap train(201710) -> val(201712) = 2 >= horizonte 2
  [ok   ] gap val(201712) -> test(201812) = 12 >= horizonte 2
  [ok   ] gap (train+val) -> test = 12 >= 2
  [ok   ] orden cronologico train < val < test
  [ok   ] m_muere (dato del futuro) no es feature
  [ok   ] el target no es feature
  [ok   ] clase_tn[i] == tn[i+2] en el producto 20001 (36 meses, 0 discrepancias)
TRAIN 10 meses (5,159 filas) · VAL [201712] · TEST [201812] · INFER [201911, 201912]


## 4 — El WAPE y los baselines

El baseline es lo que se le regala al modelo: la estimación del **nivel**, que ya
funciona. El modelo después sólo tiene que aprender la **desviación**.

El más interesante es `lineal`: una Ridge sobre los lags, o sea **un promedio móvil con
los pesos aprendidos en vez de fijos**. Es la versión formal de la regresión stepwise que
dio 0,23, y por eso está acá como candidato de primera clase y no como curiosidad.

Ridge y no mínimos cuadrados puros porque los lags están muy correlacionados entre sí
(`tn_lag1` y `tn_lag2` se parecen mucho): sin regularizar, los coeficientes se vuelven
grandes y de signos alternados, y el modelo deja de generalizar.

In [17]:
def wape(y_real, y_pred, ids=None) -> float:
    """WAPE en toneladas, agregando por producto. Identico al del pipe."""
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if ids is not None:
        _, inv = np.unique(np.asarray(ids), return_inverse=True)
        yr, yp = np.bincount(inv, weights=yr), np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def bloque(meses):
    return sup.filter(pl.col("periodo").is_in(meses))


tr, va, te = bloque(MESES_TRAIN), bloque(MESES_VAL), bloque(MESES_TEST)
COLS_LIN = ["tn"] + [f"tn_lag{k}" for k in range(1, L + 1)] + ["tn_ma3", "tn_ma6"]


def X_lin(b):
    """Matriz para la parte lineal: los niveles recientes, con los nulos del arranque en 0."""
    return b.select(COLS_LIN).fill_null(0.0).to_numpy()


def wape_de(b, pred):
    return wape(b["clase_tn"].to_numpy(), pred, b["product_id"].to_numpy())


# ── Los baselines ────────────────────────────────────────────────────────
def baseline_fijo(b, cual):
    if cual == "tn0":
        return b["tn"].to_numpy().astype(np.float64)
    if cual == "ma_pond":
        return (0.5 * b["tn"].fill_null(0).to_numpy()
                + 0.3 * b["tn_lag1"].fill_null(0).to_numpy()
                + 0.2 * b["tn_lag2"].fill_null(0).to_numpy())
    return b[f"tn_{cual}"].fill_null(0.0).to_numpy().astype(np.float64)


_ridge_base = Ridge(alpha=PARAM['ridge_alpha'])
_ridge_base.fit(X_lin(tr), tr["clase_tn"].to_numpy())


def baseline_de(b, cual, ridge=None):
    """Prediccion del baseline para el bloque b. Siempre >= 0."""
    if cual == "lineal":
        r = ridge if ridge is not None else _ridge_base
        return np.maximum(r.predict(X_lin(b)), 0.0)
    return np.maximum(baseline_fijo(b, cual), 0.0)


CANDIDATOS = ["tn0", "ma3", "ma6", "ma12", "ma_pond", "lineal"]
print(f"{'baseline':10s} {'WAPE val':>10s}")
print("-" * 22)
wape_base = {}
for c in CANDIDATOS:
    wape_base[c] = wape_de(va, baseline_de(va, c))
    print(f"{c:10s} {wape_base[c]:10.5f}")

BASELINE = (PARAM['baseline'] if PARAM['baseline'] != 'auto'
            else min(wape_base, key=wape_base.get))
print(f"\nbaseline elegido: {BASELINE}"
      + ("  (por validacion)" if PARAM['baseline'] == 'auto' else "  (forzado)"))

# ── Los pesos que aprendio la Ridge: el promedio movil optimizado ────────
_co = dict(zip(COLS_LIN, _ridge_base.coef_))
print(f"\npesos de la Ridge (intercepto {_ridge_base.intercept_:+.3f}):")
for k, v in sorted(_co.items(), key=lambda kv: -abs(kv[1]))[:8]:
    print(f"   {k:10s} {v:+.4f}")
print("Si los pesos son positivos y decrecientes, la Ridge encontro sola un promedio")
print("movil ponderado. Si alternan de signo, esta capturando reversion a la media.")

baseline     WAPE val
----------------------
tn0           0.22584
ma3           0.35063
ma6           0.33896
ma12          0.30352
ma_pond       0.29152
lineal        0.44090

baseline elegido: ma_pond  (forzado)

pesos de la Ridge (intercepto +0.493):
   tn         +1.0305
   tn_ma3     -0.6111
   tn_lag6    +0.4854
   tn_lag8    -0.4359
   tn_lag5    +0.4174
   tn_ma6     -0.3689
   tn_lag2    +0.3528
   tn_lag9    -0.2862
Si los pesos son positivos y decrecientes, la Ridge encontro sola un promedio
movil ponderado. Si alternan de signo, esta capturando reversion a la media.


## 5 — Los seis esquemas

Todos con las mismas features, la misma partición y la misma semilla. La única
diferencia es **quién se queda con qué parte del problema**.

El WAPE se mide siempre sobre **toneladas reconstruidas**, así que los seis números son
directamente comparables entre sí y con los del pipe.

In [18]:
PARAMS_LGBM = dict(objective="regression", metric="mae", verbosity=-1,
                   n_estimators=500, learning_rate=0.05, num_leaves=63,
                   min_child_samples=20, subsample=0.9, subsample_freq=1,
                   colsample_bytree=0.8, seed=PARAM['semilla'], n_jobs=-1,
                   deterministic=True, force_row_wise=True)


def fit_lgbm(b, target, params=None, lineal=False):
    p = dict(params or PARAMS_LGBM)
    if lineal:
        # linear_tree ajusta una REGRESION LINEAL dentro de cada hoja: le da al arbol
        # la capacidad de extrapolar que por construccion no tiene.
        p.update(linear_tree=True, linear_lambda=1.0)
    m = lgb.LGBMRegressor(**p)
    bp = b.to_pandas()
    m.fit(bp[FEATURES], bp[target].to_numpy() if hasattr(bp[target], "to_numpy") else bp[target],
          categorical_feature=CATS)
    return m


def evaluar_esquemas(meses_fit, b_eval):
    """Devuelve {esquema: pred_tn} para el bloque de evaluacion."""
    fit = bloque(meses_fit)
    base_fit = baseline_de(fit, BASELINE)
    base_ev = baseline_de(b_eval, BASELINE)
    ev = b_eval.to_pandas()
    out, modelos = {}, {}

    # A) el baseline solo
    out["A_baseline"] = base_ev

    # B) LightGBM al nivel
    mB = fit_lgbm(fit, "clase_tn")
    out["B_lgbm_nivel"] = mB.predict(ev[FEATURES]); modelos["B_lgbm_nivel"] = mB

    # C) lineal al nivel (Ridge sobre TODAS las features numericas, no solo los lags)
    _num = [c for c in FEATURES if c not in CATS]
    mC = Ridge(alpha=PARAM['ridge_alpha'])
    mC.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
    out["C_lineal_nivel"] = mC.predict(b_eval.select(_num).fill_null(0.0).to_numpy())
    modelos["C_lineal_nivel"] = mC

    # D) LightGBM al residuo del baseline
    fit_d = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_fit))
    mD = fit_lgbm(fit_d, "y_res")
    out["D_lgbm_residuo"] = base_ev + mD.predict(ev[FEATURES]); modelos["D_lgbm_residuo"] = mD

    # E) lineal para el nivel + LightGBM para lo que sobra
    #    Es la apuesta: la recta hace el nivel (extrapola), el arbol las interacciones.
    rE = Ridge(alpha=PARAM['ridge_alpha'])
    rE.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    base_lin_fit = np.maximum(rE.predict(X_lin(fit)), 0.0)
    base_lin_ev = np.maximum(rE.predict(X_lin(b_eval)), 0.0)
    fit_e = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_lin_fit))
    mE = fit_lgbm(fit_e, "y_res")
    out["E_lineal_mas_lgbm"] = base_lin_ev + mE.predict(ev[FEATURES])
    modelos["E_lineal_mas_lgbm"] = (rE, mE)

    # F) LightGBM con hojas lineales, al nivel
    mF = fit_lgbm(fit, "clase_tn", lineal=True)
    out["F_lgbm_hojas_lineales"] = mF.predict(ev[FEATURES]); modelos["F_lgbm_hojas_lineales"] = mF

    return out, modelos


t0 = time.time()
pred_val, mod_val = evaluar_esquemas(MESES_TRAIN, va)
print(f"[{time.time()-t0:.0f}s]\n")

ESQUEMAS = list(pred_val)
print(f"{'esquema':24s} {'WAPE val':>10s}")
print("-" * 36)
wape_val = {}
for e in ESQUEMAS:
    wape_val[e] = wape_de(va, pred_val[e])
    print(f"{e:24s} {wape_val[e]:10.5f}")

ESQUEMA = (PARAM['esquema'] if PARAM['esquema'] != 'auto'
           else min(wape_val, key=wape_val.get))
print(f"\nesquema elegido: {ESQUEMA}"
      + ("  (por validacion)" if PARAM['esquema'] == 'auto' else "  (forzado)"))
_mej = 100 * (wape_val['A_baseline'] - wape_val[ESQUEMA]) / wape_val['A_baseline']
print(f"mejora sobre el baseline solo: {_mej:+.1f}%")

[12s]

esquema                    WAPE val
------------------------------------
A_baseline                  0.29152
B_lgbm_nivel                0.32699
C_lineal_nivel              0.45910
D_lgbm_residuo              0.30532
E_lineal_mas_lgbm           0.34542
F_lgbm_hojas_lineales       0.43442

esquema elegido: B_lgbm_nivel  (forzado)
mejora sobre el baseline solo: -12.2%


## 6 — Optuna sobre el esquema ganador

Sólo se afina el esquema que ganó en validación, y **lo que se minimiza es el WAPE en
toneladas reconstruidas**, nunca el error del residuo. Si el esquema ganador es
`A_baseline` no hay nada que optimizar y esta celda lo saltea — que sería, en sí, el
resultado más interesante posible: que ningún modelo le gana al promedio.

In [19]:
def espacio(trial):
    return dict(
        objective="regression", metric="mae", verbosity=-1,
        seed=PARAM['semilla'], n_jobs=-1, subsample_freq=1,
        deterministic=True, force_row_wise=True,
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        n_estimators=trial.suggest_int("n_estimators", 200, PARAM['techo_arboles']),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )


def predecir_esquema(esquema, meses_fit, b_eval, params, semilla=None):
    """Entrena el esquema con `params` y devuelve las toneladas predichas."""
    fit = bloque(meses_fit)
    ev = b_eval.to_pandas()
    p = dict(params)
    if semilla is not None:
        p["seed"] = semilla

    if esquema == "A_baseline":
        return baseline_de(b_eval, BASELINE), None

    if esquema == "C_lineal_nivel":
        _num = [c for c in FEATURES if c not in CATS]
        r = Ridge(alpha=PARAM['ridge_alpha'])
        r.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
        return r.predict(b_eval.select(_num).fill_null(0.0).to_numpy()), r

    if esquema == "B_lgbm_nivel":
        m = fit_lgbm(fit, "clase_tn", p)
        return m.predict(ev[FEATURES]), m

    if esquema == "F_lgbm_hojas_lineales":
        m = fit_lgbm(fit, "clase_tn", p, lineal=True)
        return m.predict(ev[FEATURES]), m

    if esquema == "D_lgbm_residuo":
        bf, be = baseline_de(fit, BASELINE), baseline_de(b_eval, BASELINE)
        f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
        m = fit_lgbm(f2, "y_res", p)
        return be + m.predict(ev[FEATURES]), m

    # E_lineal_mas_lgbm
    r = Ridge(alpha=PARAM['ridge_alpha'])
    r.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    bf = np.maximum(r.predict(X_lin(fit)), 0.0)
    be = np.maximum(r.predict(X_lin(b_eval)), 0.0)
    f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
    m = fit_lgbm(f2, "y_res", p)
    return be + m.predict(ev[FEATURES]), (r, m)


MEJORES = dict(
    objective="regression", metric="mae", verbosity=-1, seed=109903, n_jobs=-1,
    subsample_freq=1, deterministic=True, force_row_wise=True,
    num_leaves=136, max_depth=8, learning_rate=0.09433176265594022,
    n_estimators=286, min_child_samples=23, subsample=0.9125472929795297,
    colsample_bytree=0.9307864807790545, reg_alpha=0.24700724500357815,
    reg_lambda=3.513103697522887e-07)
study = None
print("Optuna omitido: parámetros ganadores congelados")


Optuna omitido: parámetros ganadores congelados


## 7 — Resultados: la tabla completa

`val` es optimista para el esquema ganador (Optuna lo minimizó). `test` es el número
honesto, medido una sola vez.

In [20]:
_corte_test = min(MESES_TEST)
_idx_test = (_corte_test // 100) * 12 + (_corte_test % 100) - PARAM['horizonte']
MESES_FIT_TEST = [p for p in periodos_sup
                  if ((p // 100) * 12 + (p % 100)) <= _idx_test] \
                 if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
_pars = MEJORES or PARAMS_LGBM

pred_test, _ = evaluar_esquemas(MESES_FIT_TEST, te)
if MEJORES:
    pred_test[ESQUEMA], _ = predecir_esquema(ESQUEMA, MESES_FIT_TEST, te, _pars)

print(f"{'esquema':24s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 48)
METRICAS = {}
for e in ESQUEMAS:
    wt = wape_de(te, pred_test[e])
    METRICAS[e] = {"val": wape_val[e], "test": wt}
    marca = "  <-" if e == ESQUEMA else ""
    print(f"{e:24s} {wape_val[e]:10.5f} {wt:10.5f}{marca}")

_a, _g = METRICAS['A_baseline']['test'], METRICAS[ESQUEMA]['test']
print(f"\nganador en test: {min(METRICAS, key=lambda e: METRICAS[e]['test'])}")
print(f"{ESQUEMA} vs baseline solo, en test: {100*(_a-_g)/_a:+.1f}%")
_br = METRICAS[ESQUEMA]['test'] - METRICAS[ESQUEMA]['val']
print(f"brecha test - val: {_br:+.5f}"
      + ("   <- sobreajuste a validacion" if _br > 0.02 else ""))

pl.DataFrame([{"esquema": e, **METRICAS[e]} for e in ESQUEMAS]) \
  .write_csv(DIR_OUT / "esquemas.csv")

# ── Donde gana cada uno: por volumen del producto ───────────────────────
# El WAPE pondera por volumen, asi que lo unico que mueve la aguja son los productos
# grandes. Este corte dice si la ventaja viene de ahi o de la cola.
det = te.select("product_id", "clase_tn").with_columns(
    pl.Series("base", pred_test["A_baseline"]),
    pl.Series("gana", pred_test[ESQUEMA]))
_q = det.group_by("product_id").agg(pl.col("clase_tn").sum().alias("tn"))
_cortes = _q["tn"].qcut(4, labels=["Q1 chico", "Q2", "Q3", "Q4 grande"], allow_duplicates=True)
_q = _q.with_columns(_cortes.alias("cuartil"))
det = det.join(_q.select("product_id", "cuartil"), on="product_id", how="left")

filas = []
for c in ["Q1 chico", "Q2", "Q3", "Q4 grande"]:
    b = det.filter(pl.col("cuartil") == c)
    if b.height < 5:
        continue
    filas.append({"cuartil": c, "n_filas": b.height,
                  "tn_real": round(float(b["clase_tn"].sum()), 1),
                  "wape_baseline": round(wape(b["clase_tn"], b["base"], b["product_id"]), 4),
                  f"wape_{ESQUEMA}": round(wape(b["clase_tn"], b["gana"], b["product_id"]), 4)})
por_vol = pl.DataFrame(filas)
print()
print(por_vol)
por_vol.write_csv(DIR_OUT / "por_cuartil_de_volumen.csv")
print("\nEl WAPE pondera por volumen: si la ventaja no esta en Q4, no va a mover el")
print("numero global aunque se vea grande en los cuartiles chicos.")

esquema                    WAPE val  WAPE test
------------------------------------------------
A_baseline                  0.29152    0.25136
B_lgbm_nivel                0.32699    0.31912  <-
C_lineal_nivel              0.45910    0.32204
D_lgbm_residuo              0.30532    0.34337
E_lineal_mas_lgbm           0.34542    0.27882
F_lgbm_hojas_lineales       0.43442    0.29831

ganador en test: A_baseline
B_lgbm_nivel vs baseline solo, en test: -27.0%
brecha test - val: -0.00787

shape: (4, 5)
┌───────────┬─────────┬─────────┬───────────────┬───────────────────┐
│ cuartil   ┆ n_filas ┆ tn_real ┆ wape_baseline ┆ wape_B_lgbm_nivel │
│ ---       ┆ ---     ┆ ---     ┆ ---           ┆ ---               │
│ str       ┆ i64     ┆ f64     ┆ f64           ┆ f64               │
╞═══════════╪═════════╪═════════╪═══════════════╪═══════════════════╡
│ Q1 chico  ┆ 164     ┆ 173.4   ┆ 0.3761        ┆ 1.1806            │
│ Q2        ┆ 164     ┆ 949.0   ┆ 0.3261        ┆ 0.6432            │
│ Q3     

## 8 — Entrenamiento final y entrega

Se reentrena el esquema ganador con **todos** los meses supervisados y se predice el mes
objetivo. Del modelo final no hay métrica honesta: la que se reporta es la de `test`.

In [21]:
MESES_TODOS = sorted(periodos_sup)
print(f"reentrenando {ESQUEMA} con {len(MESES_TODOS)} meses "
      f"({MESES_TODOS[0]}..{MESES_TODOS[-1]})")

preds = []
for sem in PARAM['semillas_ensemble']:
    p, _ = predecir_esquema(ESQUEMA, MESES_TODOS, infer, _pars, semilla=sem)
    preds.append(p)
    print(f"  semilla {sem} lista")
pred_infer_tn = np.maximum(np.mean(preds, axis=0), PARAM['clip_min'])

pred_infer = infer.select("product_id", "periodo", "periodo_objetivo").with_columns(
    pl.Series("tn_pred", pred_infer_tn),
    pl.Series("baseline", baseline_de(infer, BASELINE)))
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\npredicciones: {pred_infer.height:,} filas")
print(pred_infer.group_by("periodo", "periodo_objetivo").len().sort("periodo"))
print(f"\ntn_pred   min {pred_infer_tn.min():.2f}   media {pred_infer_tn.mean():.2f}   "
      f"max {pred_infer_tn.max():.2f}")
print(f"correlacion con el baseline: "
      f"{np.corrcoef(pred_infer_tn, pred_infer['baseline'].to_numpy())[0,1]:.4f}")
print("Si esa correlacion es ~1, el modelo esta repitiendo el baseline y no aporta nada.")

reentrenando B_lgbm_nivel con 34 meses (201701..201910)
  semilla 109903 lista
  semilla 109927 lista
  semilla 203999 lista

predicciones: 1,560 filas
shape: (2, 3)
┌─────────┬──────────────────┬─────┐
│ periodo ┆ periodo_objetivo ┆ len │
│ ---     ┆ ---              ┆ --- │
│ i64     ┆ i64              ┆ u32 │
╞═════════╪══════════════════╪═════╡
│ 201911  ┆ 202001           ┆ 780 │
│ 201912  ┆ 202002           ┆ 780 │
└─────────┴──────────────────┴─────┘

tn_pred   min 0.00   media 36.34   max 1380.24
correlacion con el baseline: 0.9828
Si esa correlacion es ~1, el modelo esta repitiendo el baseline y no aporta nada.


## 9 — El CSV y el submit

In [22]:
OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")

por_producto = obj.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")
submit = oficiales.select("product_id").join(por_producto, on="product_id", how="left")
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")

print(f"mes objetivo {OBJ}: {obj.height} filas -> {por_producto.height} productos")
print(f"lista oficial: {oficiales.height}   sin prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista. Revisalo antes de subir.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10))

path_submit = DIR_OUT / f"submission_{OBJ}.csv"
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"\nGuardado: {path_submit}")


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("\nPARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    flag_submit = DIR_OUT / "submit_v4.done"
    if flag_submit.exists():
        print("\nSubmit v4 ya realizado; se evita duplicarlo.")
    kd = Path.home() / ".kaggle" / "kaggle.json"
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kd); kd.chmod(0o600); break
    if flag_submit.exists():
        pass
    elif not kd.exists():
        print("\nSin credenciales de Kaggle. El CSV ya esta generado.")
    else:
        kd.chmod(0o600)
        msg = PARAM['mensaje_submit'] or (
            f"{ESQUEMA} sobre {BASELINE} | wape_test={METRICAS[ESQUEMA]['test']:.5f}")
        ok, salida = kaggle_cli(["competitions", "submit",
                                 "-c", PARAM['kaggle_competition'],
                                 "-f", str(path_submit), "-m", msg])
        print(f"\nmensaje: {msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")
        if ok:
            flag_submit.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))

mes objetivo 202002: 780 filas -> 780 productos
lista oficial: 780   sin prediccion (van en 0): 0

tn   min 0.000   media 38.454   max 1380.238   suma 29,994.3
shape: (10, 2)
┌────────────┬─────────────┐
│ product_id ┆ tn          │
│ ---        ┆ ---         │
│ i64        ┆ f64         │
╞════════════╪═════════════╡
│ 20001      ┆ 1138.656246 │
│ 20002      ┆ 1380.237831 │
│ 20003      ┆ 694.377855  │
│ 20004      ┆ 585.531706  │
│ 20005      ┆ 638.897399  │
│ 20006      ┆ 430.388947  │
│ 20007      ┆ 422.376207  │
│ 20008      ┆ 376.03452   │
│ 20009      ┆ 591.204404  │
│ 20010      ┆ 392.861002  │
└────────────┴─────────────┘

Guardado: /home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v10_balanceada_v4/submission_202002.csv

Submit v4 ya realizado; se evita duplicarlo.


## 10 — Ensemble interno con la v3

Usa exclusivamente `PATH_V3_FINAL`, generado por la Parte A en esta misma ejecución. Genera tres mezclas para diagnóstico y no las envía.


In [23]:
# Ensemble de dos modelos con errores distintos
path_v3 = PATH_V3_FINAL  # generado arriba en esta misma ejecución
DIR_ENS = DIR_OUT / "ensembles"
DIR_ENS.mkdir(parents=True, exist_ok=True)

if not path_v3.exists():
    print(f"Todavia no existe {path_v3}. La v4 pura ya quedo generada.")
else:
    v3 = pl.read_csv(path_v3).rename({"tn": "tn_v3"})
    v4 = submit.rename({"tn": "tn_v4"})
    ambos = v3.join(v4, on="product_id", how="inner")
    if ambos.height != oficiales.height:
        raise RuntimeError(f"Ensemble incompleto: {ambos.height}/{oficiales.height} productos")

    for peso_v4 in (0.25, 0.50, 0.75):
        peso_v3 = 1.0 - peso_v4
        ens = ambos.select(
            "product_id",
            (pl.col("tn_v3") * peso_v3 + pl.col("tn_v4") * peso_v4)
              .clip(lower_bound=0.0).alias("tn")
        ).sort("product_id")
        nombre = f"submission_ens_v3_{peso_v3:.2f}_v4_{peso_v4:.2f}.csv"
        ruta = DIR_ENS / nombre
        ens.write_csv(ruta)
        print(f"{nombre}: suma={ens['tn'].sum():,.1f}")

        if PARAM['submit_ensembles']:
            flag = DIR_ENS / f"{nombre}.done"
            if flag.exists():
                print("  ya enviado")
                continue
            ok, salida = kaggle_cli(["competitions", "submit",
                                     "-c", PARAM['kaggle_competition'],
                                     "-f", str(ruta),
                                     "-m", f"ensemble v3={peso_v3:.2f} v4={peso_v4:.2f}"])
            print("  " + ("enviado" if ok else "fallo") + ": " + salida[-200:])
            if ok:
                flag.write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


submission_ens_v3_0.75_v4_0.25.csv: suma=29,265.1
  ya enviado
submission_ens_v3_0.50_v4_0.50.csv: suma=29,508.2
  ya enviado
submission_ens_v3_0.25_v4_0.75.csv: suma=29,751.2
  ya enviado


## 11 — Registro y leaderboard

Una fila por experimento en `exp_residuo/leaderboard_residuo.csv`, con **el WAPE de los
seis esquemas en cada corrida**. Así se ve si el ranking entre esquemas es estable al
cambiar el baseline o las features, que es lo que decide si el hallazgo es real.

In [24]:
resultado = {
    'experimento': EXPERIMENTO,
    'idea': 'residuo sobre baseline: el nivel lo hace el baseline, la desviacion el modelo',
    'granularidad': 'producto-mes',
    'baseline_elegido': BASELINE, 'wape_baselines_val': wape_base,
    'esquema_elegido': ESQUEMA, 'metricas_por_esquema': METRICAS,
    'horizonte': H, 'max_lags': L,
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'n_features': len(FEATURES), 'features': FEATURES,
    'n_trials': len(study.trials) if study else 0,
    'hiperparametros': study.best_params if study else {},
    'pesos_ridge_baseline': {k: round(float(v), 5) for k, v in _co.items()},
    'ridge_alpha': PARAM['ridge_alpha'],
    'n_sin_prediccion': sin_pred,
    'tn_total': float(submit['tn'].sum()),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {'experimento': EXPERIMENTO, 'baseline': BASELINE, 'esquema': ESQUEMA,
        'max_lags': L, 'n_features': len(FEATURES),
        'wape_test_ganador': round(METRICAS[ESQUEMA]['test'], 5),
        'wape_val_ganador': round(METRICAS[ESQUEMA]['val'], 5),
        **{f"test_{e}": round(METRICAS[e]['test'], 5) for e in ESQUEMAS},
        'mejora_vs_baseline_pct': round(100*(_a-_g)/_a, 2),
        'sin_prediccion': sin_pred, 'tn_total': round(float(submit['tn'].sum()), 1)}

path_lb = RUTA_EXP / "leaderboard_residuo.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("wape_test_ganador").write_csv(path_lb)

print(f"Archivos en {DIR_OUT.relative_to(BUCKET)}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_residuo.csv ({nueva.height} experimentos):")
print(nueva.select("baseline", "esquema", "wape_test_ganador",
                   "test_A_baseline", "mejora_vs_baseline_pct"))

Archivos en exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v10_balanceada_v4:
  - ensembles
  - esquemas.csv
  - por_cuartil_de_volumen.csv
  - predicciones_inferencia.parquet
  - resultado.json
  - submission_202002.csv
  - submit_v4.done

leaderboard_residuo.csv (5 experimentos):
shape: (5, 5)
┌──────────┬──────────────┬───────────────────┬─────────────────┬────────────────────────┐
│ baseline ┆ esquema      ┆ wape_test_ganador ┆ test_A_baseline ┆ mejora_vs_baseline_pct │
│ ---      ┆ ---          ┆ ---               ┆ ---             ┆ ---                    │
│ str      ┆ str          ┆ f64               ┆ f64             ┆ f64                    │
╞══════════╪══════════════╪═══════════════════╪═════════════════╪════════════════════════╡
│ tn0      ┆ A_baseline   ┆ 0.26617           ┆ 0.26617         ┆ 0.0                    │
│ ma_pond  ┆ B_lgbm_nivel ┆ 0.26717           ┆ 0.33219         ┆ 19.57                  │
│ ma_pond  ┆ B_lgbm_nivel 

## Cómo leer el resultado

La tabla de la sección 7 contesta la pregunta directamente. Cuatro desenlaces posibles,
y ninguno es un fracaso:

| Si gana… | Significa |
|---|---|
| **A_baseline** | Ningún modelo le gana a un promedio. Es un resultado fuerte y publicable: en esta serie el ruido domina y la complejidad no compra nada. |
| **C_lineal_nivel** | Se confirma tu observación: la señal es lineal en los niveles recientes, y el GBM está de más. |
| **E_lineal_mas_lgbm** | La hipótesis del notebook: la recta hace el nivel y el árbol agrega algo sobre el residuo. El mejor de los dos mundos. |
| **B** o **F** | El GBM sí puede con el nivel, y lo de la regresión lineal era un problema de tuneo, no estructural. |

Y hay un número que conviene mirar antes de festejar: **la correlación entre la
predicción final y el baseline**, que imprime la sección 8. Si es 0,99, el modelo está
repitiendo el promedio con pasos extra — el WAPE puede mejorar un poco y aun así no
haber aprendido nada.

### Qué variar después

1. **El baseline.** `ma3` contra `lineal` es la comparación que más importa, porque
   `lineal` *es* el promedio con pesos aprendidos.
2. **`max_lags`.** Si la Ridge le pone peso a `tn_lag11` y `tn_lag12`, hay estacionalidad
   anual y conviene subirlo.
3. **`ridge_alpha`.** Con lags muy correlacionados, este número decide si los
   coeficientes salen estables o alternando de signo. Mirá los pesos que imprime la
   sección 4.

In [25]:
# CHECKPOINT EXPLÍCITO DEL CAMPEÓN v3/v4
PATH_CAMPEON = DIR_ENS / "submission_ens_v3_0.75_v4_0.25.csv"
if not PATH_CAMPEON.exists():
    raise FileNotFoundError(f"No se generó el ensemble interno {PATH_CAMPEON}")
pred_campeon_visible = pd.read_csv(PATH_CAMPEON).sort_values("product_id").reset_index(drop=True)
print("Campeón v3/v4 listo:", PATH_CAMPEON)
print("filas=", len(pred_campeon_visible), "tn total=", pred_campeon_visible["tn"].sum())


Campeón v3/v4 listo: /home/ds/buckets/b1/exp_residuo/residuo_p_12lags_base-ma_pond_esq-B_lgbm_nivel_val201712-201712_test201812_v10_balanceada_v4/ensembles/submission_ens_v3_0.75_v4_0.25.csv
filas= 780 tn total= 29265.135103008535


# Parte C — Ridge visible

Ridge se entrena desde `sell-in.txt.gz`, restringido al universo solicitado, sobre un panel
mensual completo con ceros. Usa nivel actual, 12 rezagos, medias móviles y mes calendario.
Es determinista (`alpha=100`) y funciona sólo como corrector conservador.


In [26]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

DIR_DATOS_RIDGE = BUCKET / "datasets"
ARCHIVO_SELLIN_RIDGE = DIR_DATOS_RIDGE / "sell-in.txt.gz"
ARCHIVO_TARGET_RIDGE = DIR_DATOS_RIDGE / "product_id_apredecir201912.txt"
print("Inputs Ridge:", ARCHIVO_SELLIN_RIDGE, ARCHIVO_TARGET_RIDGE, sep="\n - ")

sell_ridge = pd.read_csv(ARCHIVO_SELLIN_RIDGE, sep="\t")
target_ridge = pd.read_csv(ARCHIVO_TARGET_RIDGE, sep="\t")["product_id"]
sell_ridge = sell_ridge[sell_ridge.product_id.isin(target_ridge)]
panel_ridge = sell_ridge.groupby(["product_id", "periodo"], as_index=False).agg(tn=("tn", "sum"))
meses_ridge = sorted(panel_ridge.periodo.unique())
grid_ridge = pd.MultiIndex.from_product(
    [target_ridge, meses_ridge], names=["product_id", "periodo"]
).to_frame(index=False)
panel_ridge = grid_ridge.merge(panel_ridge, on=["product_id", "periodo"], how="left")
panel_ridge["tn"] = panel_ridge["tn"].fillna(0.0)
panel_ridge = panel_ridge.sort_values(["product_id", "periodo"])
print("Panel Ridge denso:", len(panel_ridge), "filas; ceros:", int((panel_ridge.tn == 0).sum()))

grupo_ridge = panel_ridge.groupby("product_id")["tn"]
for lag in range(1, 13):
    panel_ridge[f"lag{lag}"] = grupo_ridge.shift(lag)
for ventana in (3, 6, 12):
    panel_ridge[f"ma{ventana}"] = grupo_ridge.transform(
        lambda x: x.rolling(ventana, min_periods=1).mean()
    )
panel_ridge["target"] = grupo_ridge.shift(-2)
panel_ridge["mes"] = panel_ridge.periodo % 100

FEATURES_RIDGE = (["tn"] + [f"lag{k}" for k in range(1, 13)]
                  + ["ma3", "ma6", "ma12", "mes"])
train_ridge = panel_ridge[panel_ridge.target.notna()].copy()
infer_ridge = panel_ridge[panel_ridge.periodo == 201912].copy()
escalador_ridge = StandardScaler().fit(train_ridge[FEATURES_RIDGE].fillna(0))
modelo_ridge = Ridge(alpha=100.0)
modelo_ridge.fit(
    escalador_ridge.transform(train_ridge[FEATURES_RIDGE].fillna(0)),
    train_ridge["target"],
)
pred_ridge = np.maximum(
    modelo_ridge.predict(escalador_ridge.transform(infer_ridge[FEATURES_RIDGE].fillna(0))), 0
)
pred_ridge_visible = infer_ridge[["product_id"]].copy()
pred_ridge_visible["tn"] = pred_ridge
pred_ridge_visible = pred_ridge_visible.sort_values("product_id").reset_index(drop=True)
print("Ridge entrenada: alpha=100; filas=", len(pred_ridge_visible),
      "tn total=", pred_ridge_visible.tn.sum())


Inputs Ridge:
 - /home/ds/buckets/b1/datasets/sell-in.txt.gz
 - /home/ds/buckets/b1/datasets/product_id_apredecir201912.txt
Panel Ridge denso: 28080 filas; ceros: 5731
Ridge entrenada: alpha=100; filas= 780 tn total= 25206.189530736796


# Parte D — Ensemble final y controles

No se busca ningún archivo por nombre. Se combinan únicamente objetos y rutas creados en
esta ejecución. Además se verifica igualdad exacta del universo de productos, ausencia de
duplicados, valores finitos, no negatividad y cantidad de filas.


In [27]:
OUT_V10 = BUCKET / "pipeline_v10_visible"
OUT_V10.mkdir(parents=True, exist_ok=True)

campeon = pred_campeon_visible.sort_values("product_id").reset_index(drop=True)
ridge_final = pred_ridge_visible.sort_values("product_id").reset_index(drop=True)
if not campeon.product_id.equals(ridge_final.product_id):
    raise RuntimeError("El universo de productos de Ridge no coincide con el campeón v3/v4")
if campeon.product_id.duplicated().any():
    raise RuntimeError("Hay product_id duplicados")

pred_base = campeon.tn.to_numpy(dtype=float)
pred_lineal = ridge_final.tn.to_numpy(dtype=float)
diferencia = np.clip(
    pred_lineal - pred_base,
    -TOPE_RIDGE * np.maximum(pred_base, 1.0),
     TOPE_RIDGE * np.maximum(pred_base, 1.0),
)
pred_final = np.maximum(pred_base + PESO_RIDGE * diferencia, 0.0)

submission_v10 = pd.DataFrame({"product_id": campeon.product_id, "tn": pred_final})
if len(submission_v10) != len(campeon) or not np.isfinite(submission_v10.tn).all():
    raise RuntimeError("Submission inválida")

PATH_SUBMISSION_V10 = OUT_V10 / "submission_v10_visible_multimodelo_multisemilla.csv"
PATH_DIAGNOSTICO_V10 = OUT_V10 / "diagnostico_v10_visible.csv"
submission_v10.to_csv(PATH_SUBMISSION_V10, index=False)
pd.DataFrame({
    "product_id": campeon.product_id,
    "campeon_v3_v4": pred_base,
    "ridge": pred_lineal,
    "correccion_recortada": diferencia,
    "pred_final": pred_final,
}).to_csv(PATH_DIAGNOSTICO_V10, index=False)

print("RESULTADO FINAL")
print(" - modelos: LightGBM v3 (3 semillas), LightGBM v4 (3 semillas), Ridge")
print(" - pesos: campeón=75% v3 + 25% v4; Ridge corrige al 10% con tope ±30%")
print(" - filas:", len(submission_v10))
print(" - tn total:", submission_v10.tn.sum())
print(" - submission:", PATH_SUBMISSION_V10)
print(" - diagnóstico:", PATH_DIAGNOSTICO_V10)


RESULTADO FINAL
 - modelos: LightGBM v3 (3 semillas), LightGBM v4 (3 semillas), Ridge
 - pesos: campeón=75% v3 + 25% v4; Ridge corrige al 10% con tope ±30%
 - filas: 780
 - tn total: 28963.98665385096
 - submission: /home/ds/buckets/b1/pipeline_v10_visible/submission_v10_visible_multimodelo_multisemilla.csv
 - diagnóstico: /home/ds/buckets/b1/pipeline_v10_visible/diagnostico_v10_visible.csv


# Parte E — Envío opcional a Kaggle

Esta es la única celda que produce una acción externa. Cambiar `SUBMIT_V10` a `True` sólo
cuando se haya revisado el diagnóstico. El archivo ya queda generado aunque se mantenga
en `False`.


In [28]:
SUBMIT_V10 = True
FLAG_SUBMIT_V10 = OUT_V10 / "submit_v10_visible.done"
if SUBMIT_V10 and not FLAG_SUBMIT_V10.exists():
    import shutil, subprocess, time
    credencial = Path.home() / ".kaggle" / "kaggle.json"
    credencial.parent.mkdir(parents=True, exist_ok=True)
    if not credencial.exists():
        for candidata in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if candidata.exists():
                shutil.copy(candidata, credencial)
                credencial.chmod(0o600)
                break
    if not credencial.exists():
        raise FileNotFoundError("No se encontró kaggle.json; el CSV sí fue generado")
    resultado_submit = subprocess.run([
        "kaggle", "competitions", "submit", "-c", "labo-iii-2026-ba",
        "-f", str(PATH_SUBMISSION_V10),
        "-m", "v10 visible multimodelo multisemilla",
    ], capture_output=True, text=True)
    print((resultado_submit.stdout or "") + (resultado_submit.stderr or ""))
    if resultado_submit.returncode == 0:
        FLAG_SUBMIT_V10.write_text(time.strftime("%F %T"))
elif FLAG_SUBMIT_V10.exists():
    print("La v10 visible ya fue enviada; se evita duplicarla")
else:
    print("SUBMIT_V10=False: CSV generado, no enviado")


La v10 visible ya fue enviada; se evita duplicarla
